# 2.3 Очистка: пропуски, дубликаты, выбросы — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 02 «Pandas и подготовка данных» · страница 2.3.

В ноутбуке четыре ячейки раздела «Код». Они выполняются по порядку: каждая продолжает таблицу `df` и журнал очистки `log` предыдущей.

**Данные.** Учебная выгрузка `data/orders_raw.csv` на 207 строк должна лежать рядом с ноутбуком в папке `data`. Её делает `gen_data.py` с зёрнами 23 и 230.

Вывод под ячейками получен на Python 3.13 и pandas 3.0. В pandas 2 строковые столбцы будут типа `object`, а не `str`.

In [1]:
import platform

import pandas as pd

print("Python", platform.python_version(), "| pandas", pd.__version__)

Python 3.13.18 | pandas 3.0.5


### Отчёт о пропусках по столбцам

In [2]:
import pandas as pd

df = pd.read_csv("data/orders_raw.csv", parse_dates=["date", "updated_at"])
log = []                      # журнал очистки: шаг, что сделано, сколько строк

report = pd.DataFrame({
    "тип": df.dtypes.astype(str),
    "пропусков": df.isna().sum(),
    "доля, %": (df.isna().mean() * 100).round(1),
}).sort_values("пропусков", ascending=False)

print("строк, столбцов:", df.shape)
print(report)
print("строк после dropna():", len(df.dropna()))
print('строк после dropna(subset=["amount"]):', len(df.dropna(subset=["amount"])))

строк, столбцов: (207, 9)
                       тип  пропусков  доля, %
channel                str         28     13.5
amount             float64         12      5.8
brand                  str          5      2.4
date        datetime64[us]          3      1.4
order_id             int64          0      0.0
country                str          0      0.0
items                int64          0      0.0
status                 str          0      0.0
updated_at  datetime64[us]          0      0.0
строк после dropna(): 163
строк после dropna(subset=["amount"]): 195


Тип str у brand, country, channel и status — строковый тип pandas 3 по умолчанию. Из 207 строк dropna() оставил 163, хотя сумм не хватает только в 12.

### Дубликаты: полные и по ключу, остаётся свежая версия

In [3]:
full = df.duplicated()                      # строка целиком повторяет одну из предыдущих
print("полных дубликатов:", full.sum(), df.loc[full, "order_id"].tolist())
df = df[~full]
log.append(("полные дубликаты", "удалены", int(full.sum())))

dup_key = df.duplicated("order_id", keep=False)   # все версии заказов с повторным id
print(df.loc[dup_key, ["order_id", "status", "updated_at"]]
        .sort_values(["order_id", "updated_at"]).head(4).to_string(index=False))

before = len(df)
df = (df.sort_values("updated_at")
        .drop_duplicates("order_id", keep="last")   # остаётся самая свежая версия
        .sort_values("order_id"))
log.append(("повторы order_id", "оставлена свежая версия", before - len(df)))
print("строк:", before, "→", len(df), "| order_id уникален:", df["order_id"].is_unique)

полных дубликатов: 3 [1047, 1113, 1178]
 order_id    status          updated_at
     1022      paid 2026-01-08 21:09:00
     1022 cancelled 2026-01-10 23:26:00
     1089      paid 2026-02-06 09:06:00
     1089 cancelled 2026-02-07 11:23:00
строк: 204 → 200 | order_id уникален: True


Сначала полные дубликаты, потом дубликаты по ключу: в обратном порядке drop_duplicates("order_id") удалил бы и полные копии, но журнал смешал бы два разных решения.

### Флаг выброса по IQR и проверка здравым смыслом

In [4]:
q1, q3 = df["amount"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["amount_outlier"] = df["amount"].notna() & ~df["amount"].between(low, high)
print(f"Q1 = {q1:.0f}, Q3 = {q3:.0f}, IQR = {iqr:.0f}, границы: {low:.0f} … {high:.0f}")

per_item = df["amount"] / df["items"]           # здравый смысл: цена одного товара
cols = ["order_id", "items", "amount"]
print(df.loc[df["amount_outlier"], cols].assign(за_товар=per_item.round()).to_string(index=False))

z = (df["amount"] - df["amount"].mean()) / df["amount"].std()
print("|z| > 3:", df.loc[z.abs() > 3, "order_id"].tolist())

typo = per_item > 50_000                         # дороже 50 000 ₽ за штуку товаров нет
df.loc[typo, "amount"] = float("nan")            # ошибка ввода: настоящей суммы не знаем
df.loc[typo, "amount_outlier"] = False
z = (df["amount"] - df["amount"].mean()) / df["amount"].std()
print("|z| > 3 после исправления:", df.loc[z.abs() > 3, "order_id"].tolist())
log.append(("ошибка ввода в amount", "заменена на NaN", int(typo.sum())))
log.append(("крупные заказы по IQR", "оставлены, флаг amount_outlier", int(df["amount_outlier"].sum())))

Q1 = 1530, Q3 = 4420, IQR = 2890, границы: -2805 … 8755
 order_id  items    amount  за_товар
     1003      5   10900.0    2180.0
     1019      3    8920.0    2973.0
     1024      5    9140.0    1828.0
     1033      4   10530.0    2632.0
     1034      5   10150.0    2030.0
     1041      4    9510.0    2378.0
     1046      1 1790000.0 1790000.0
     1113      3    9820.0    3273.0
     1122      5   10350.0    2070.0
     1158      3   11300.0    3767.0
     1169      5   10370.0    2074.0
     1192      4   14440.0    3610.0
|z| > 3: [1046]
|z| > 3 после исправления: [1003, 1158, 1192]


Нижняя граница IQR отрицательна, а суммы меньше нуля не бывают, поэтому правило работает только сверху. Ошибкой признан один заказ, 1046; флаг amount_outlier остался у одиннадцати настоящих крупных заказов.

### Нормализация брендов и журнал очистки

In [5]:
raw = df["brand"]
brand = (raw.str.strip()
            .str.lower()
            .str.replace(r"[^а-яёa-z0-9\s]", "", regex=True)  # всё, кроме букв и цифр
            .str.replace(r"\s+", " ", regex=True)             # двойные пробелы
            .str.strip()
            .str.capitalize())
print("написаний бренда:", raw.nunique(), "→", brand.nunique())
print(raw[brand == "Луч"].unique().tolist())

changed = raw.notna() & (raw != brand)          # NaN != NaN, поэтому пустые отсекаем
log.append(("написание бренда", "strip, lower, регулярные выражения", int(changed.sum())))
log.append(("пустой бренд", "«неизвестно»", int(brand.isna().sum())))
log.append(("пустой канал", "«неизвестно»", int(df["channel"].isna().sum())))
log.append(("пропуск в amount", "флаг amount_missing", int(df["amount"].isna().sum())))
df["brand"] = brand.fillna("неизвестно")
df["channel"] = df["channel"].fillna("неизвестно")
df["amount_missing"] = df["amount"].isna()       # пометить, а не заполнить

print(df["brand"].value_counts().to_dict())
print(pd.DataFrame(log, columns=["шаг", "что сделано", "строк"]).to_string(index=False))

написаний бренда: 29 → 4
['Луч.', 'Луч!', 'Луч ', 'Луч', ' Луч', 'луч', 'луч  ', '«Луч»']
{'Сова': 59, 'Луч': 51, 'Кедр': 44, 'Бриз': 41, 'неизвестно': 5}
                  шаг                        что сделано  строк
     полные дубликаты                            удалены      3
     повторы order_id            оставлена свежая версия      4
ошибка ввода в amount                    заменена на NaN      1
крупные заказы по IQR     оставлены, флаг amount_outlier     11
     написание бренда strip, lower, регулярные выражения     74
         пустой бренд                       «неизвестно»      5
         пустой канал                       «неизвестно»     26
     пропуск в amount                флаг amount_missing     13


Буквы в шаблоне перечислены явно не случайно. В pandas 3 строки хранит pyarrow, и регулярное выражение выполняет его движок, а не модуль re: в нём \w — только латиница, цифры и подчёркивание. Шаблон [^\w\s] в нашей проверке стёр бы бренды целиком, а на столбце типа object сработал бы как обычно.